# Fabric Defect Inspector: train YOLO on the Textile dataset

Classes: **Hole, Knot, Line, Stain**. Data: `prepared/textile_yolo` made by `prepare_data.py` (split by original photo, so the test set is clean).

**Before you start**
1. On your PC, zip the folder `C:\Users\mehdi\Desktop\Fabric\prepared\textile_yolo` (right-click > Send to > Compressed folder) so you get `textile_yolo.zip`.
2. Upload `textile_yolo.zip` to the top level of your Google Drive (My Drive).
3. In Colab: Runtime > Change runtime type > **T4 GPU**. Then run the cells from top to bottom.

## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU. Use Runtime > Change runtime type > T4 GPU, then run again."
print("GPU:", torch.cuda.get_device_name(0))

## 2. Install Ultralytics

In [ ]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

## 3. Get the dataset from Google Drive
Change `ZIP_PATH` if you put the zip somewhere else.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ZIP_PATH = '/content/drive/MyDrive/textile_yolo.zip'
import os
assert os.path.exists(ZIP_PATH), f"Not found: {ZIP_PATH}. Upload textile_yolo.zip to My Drive or fix ZIP_PATH."

## 4. Unzip and fix the paths
`data.yaml` was written on Windows with a `C:\\...` path. This cell points it at the Colab folder instead.

In [ ]:
import zipfile, pathlib, yaml, shutil

DATA_ROOT = pathlib.Path('/content/data')
shutil.rmtree(DATA_ROOT, ignore_errors=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    # Windows zips can store paths with backslashes; normalise them
    for info in z.infolist():
        name = info.filename.replace('\\', '/')
        if name.endswith('/'):
            continue
        target = DATA_ROOT / name
        target.parent.mkdir(parents=True, exist_ok=True)
        with z.open(info) as src, open(target, 'wb') as dst:
            shutil.copyfileobj(src, dst)

yamls = list(DATA_ROOT.rglob('data.yaml'))
assert yamls, "data.yaml not found inside the zip. Zip the textile_yolo folder itself."
DATA_YAML = yamls[0]
DS = DATA_YAML.parent

cfg = yaml.safe_load(open(DATA_YAML))
cfg['path'] = str(DS)
cfg['train'], cfg['val'], cfg['test'] = 'train/images', 'valid/images', 'test/images'
yaml.safe_dump(cfg, open(DATA_YAML, 'w'), sort_keys=False)
print(open(DATA_YAML).read())

## 5. Check the data
Expected: train 1488 images, valid 280, test 140.

In [ ]:
import collections
names = cfg['names']
for split in ('train', 'valid', 'test'):
    imgs = list((DS / split / 'images').glob('*'))
    labs = list((DS / split / 'labels').glob('*.txt'))
    counts = collections.Counter()
    for l in labs:
        for line in open(l):
            if line.strip():
                counts[names[int(line.split()[0])]] += 1
    print(f"{split:5s}: {len(imgs):4d} images, {len(labs):4d} label files, boxes {dict(counts)}")
    assert len(imgs) == len(labs) and len(imgs) > 0, f"{split}: images and labels do not match"

## 6. Train
`yolo11s` for 60 epochs at 640 px. Takes roughly 15 to 30 minutes on a T4. Training stops early if validation does not improve for 15 epochs.

If recall on thin defects (Line, Knot) is low afterwards, re-run with `IMGSZ = 960` and `BATCH = 8`.

In [ ]:
from ultralytics import YOLO

IMGSZ, BATCH, EPOCHS = 640, 16, 60
model = YOLO('yolo11s.pt')
results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, patience=15,
    project='/content/runs', name='textile_yolo11s', exist_ok=True, seed=0,
)
RUN = pathlib.Path('/content/runs/textile_yolo11s')
BEST = RUN / 'weights' / 'best.pt'
print("best weights:", BEST)

## 7. Test-set results
These are the numbers for your report: the model has never seen these photos.

In [ ]:
best = YOLO(str(BEST))
m = best.val(data=str(DATA_YAML), split='test', imgsz=IMGSZ, project='/content/runs', name='textile_test', exist_ok=True)

import pandas as pd
rows = [{'class': 'all', 'precision': m.box.mp, 'recall': m.box.mr, 'mAP50': m.box.map50, 'mAP50-95': m.box.map}]
for i, c in enumerate(m.box.ap_class_index):
    p, r, ap50, ap = m.box.class_result(i)
    rows.append({'class': best.names[int(c)], 'precision': p, 'recall': r, 'mAP50': ap50, 'mAP50-95': ap})
table = pd.DataFrame(rows).round(3)
table.to_csv('/content/runs/test_metrics.csv', index=False)
table

## 8. Training curves and confusion matrix

In [ ]:
from IPython.display import Image, display
for f in (RUN / 'results.png', pathlib.Path('/content/runs/textile_test/confusion_matrix_normalized.png')):
    if f.exists():
        print(f.name); display(Image(filename=str(f), width=900))

## 9. Look at predictions on test images

In [ ]:
import random
test_imgs = sorted((DS / 'test' / 'images').glob('*'))
random.seed(1)
pred = best.predict([str(p) for p in random.sample(test_imgs, 6)], imgsz=IMGSZ, conf=0.25, save=True,
                    project='/content/runs', name='textile_pred', exist_ok=True, verbose=False)
for f in sorted(pathlib.Path('/content/runs/textile_pred').glob('*.jpg'))[:6]:
    display(Image(filename=str(f), width=420))

## 10. Save the results
Copies `best.pt`, the metrics table and the plots to Google Drive, and downloads `best.pt` to your PC. Put `best.pt` in your Fabric folder.

In [ ]:
OUT = pathlib.Path('/content/drive/MyDrive/fabric_inspector_yolo')
OUT.mkdir(exist_ok=True)
shutil.copy(BEST, OUT / 'best.pt')
shutil.copy('/content/runs/test_metrics.csv', OUT / 'test_metrics.csv')
for f in list(RUN.glob('*.png')) + list(RUN.glob('*.csv')) + list(pathlib.Path('/content/runs/textile_test').glob('*.png')):
    shutil.copy(f, OUT / f.name)
print("saved to", OUT)

from google.colab import files
files.download(str(BEST))